In [ ]:
"""S003_Envelhecimento_Populacional_Estado_Nutricional.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/10qaQpMrsqJ_dweSAPRB_1g5E_1KuBIta

# Análise do Envelhecimento Populacional e do Estado Nutricional da Pessoa Idosa no Brasil por meio de Dados Públicos e PLN

## Introdução

O envelhecimento populacional é um processo que pode ser analisado a partir de diferentes fontes de informação, especialmente quando são considerados aspectos demográficos e relacionados ao estado nutricional da população idosa. Para compreender esse cenário de forma estruturada, é necessário reunir informações provenientes de diferentes bases e fontes públicas.

Este projeto propõe a construção de um **corpus sobre o envelhecimento populacional no Brasil e o estado nutricional da população com 60 anos ou mais**, utilizando dados públicos, informações institucionais e literatura científica. O corpus integra diferentes tipos de fontes, incluindo dados do **IBGE/SIDRA**, informações relacionadas ao **SISVAN**, conteúdos institucionais do **IBGE e do Ministério da Saúde**, além de literatura científica.

A construção do corpus envolve etapas de **coleta por APIs públicas, web scraping, organização, limpeza, deduplicação e geração de metadados**. Após essa etapa, os dados são preparados para aplicação de técnicas de **Processamento de Linguagem Natural (PLN)**.

Como parte da análise, os documentos são submetidos ao pré-processamento em português e posteriormente representados utilizando a técnica **TF-IDF**. A partir dessa representação, é realizada uma busca por similaridade, permitindo analisar a relação entre consultas e documentos presentes no corpus.

Dessa forma, o projeto busca demonstrar, de maneira prática, como diferentes fontes de dados podem ser organizadas em um corpus temático e utilizadas em um fluxo de PLN, desde a coleta e preparação dos dados até a recuperação e avaliação dos documentos.

### Objetivo geral

Construir e analisar um corpus relacionado ao envelhecimento populacional e ao estado nutricional da população idosa no Brasil, integrando diferentes fontes de dados públicos e aplicando técnicas de Processamento de Linguagem Natural para organização, representação e recuperação de informações.

### Etapas principais

1. Preparação do ambiente e organização das fontes;
2. Coleta de dados demográficos e nutricionais;
3. Coleta de literatura científica;
4. Extração de informações de páginas oficiais;
5. Organização e deduplicação do corpus;
6. Geração e validação dos metadados;
7. Pré-processamento dos textos em português;
8. Representação dos documentos utilizando TF-IDF;
9. Busca por similaridade;
10. Avaliação dos resultados;
11. Ampliação do corpus por meio de APIs públicas;
12. Reprocessamento e análise das limitações do projeto.

## Mapa de execução

| Bloco | Objetivo |
|---|---|
| 1–4 | Preparação, estrutura e fontes |
| 5–9 | Coleta por APIs |
| 10–15 | Web scraping |
| 16–28 | Organização, limpeza, metadados e corpus |
| 29–35 | PLN, busca, avaliação e documentação |
| 36 | Ampliação do corpus por APIs |
| 37 | Síntese final |

> **Ordem recomendada:** execute as células de cima para baixo em uma sessão nova do Colab. As etapas posteriores dependem das variáveis produzidas nas etapas anteriores.

## 1. Preparação do ambiente


!pip install -q requests beautifulsoup4 pandas scikit-learn nltk matplotlib seaborn

"""## 2. Importação das bibliotecas"""

from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import hashlib
import json
import re
import shutil
import time
import unicodedata

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("rslp", quiet=True)

print("Bibliotecas importadas e recursos do NLTK preparados.")

"""## 3. Estrutura do projeto"""

raiz = Path("EnvelhecimentoPopulacionalBrasil")

pastas = [
    "dados_brutos/ibge",
    "dados_brutos/sisvan",
    "dados_brutos/datasus",
    "dados_brutos/pubmed",
    "dados_brutos/crossref",
    "dados_brutos/web_scraping",
    "dados_processados",
    "metadados",
    "logs",
    "resultados"
]

for pasta in pastas:
    (raiz / pasta).mkdir(parents=True, exist_ok=True)

for caminho in sorted(raiz.rglob("*")):
    if caminho.is_dir():
        print(caminho)

"""## 4. Configurações e fontes"""

HEADERS = {
    "User-Agent": (
        "EnvelhecimentoPopulacionalBrasil/1.0 "
        "(projeto acadêmico de PLN sobre envelhecimento e estado nutricional)"
    )
}

PAUSA_ENTRE_REQUISICOES = 1
TIMEOUT = 30

URL_IBGE_SIDRA = "https://apisidra.ibge.gov.br/values"
URL_IBGE_CENSO = "https://www.ibge.gov.br/estatisticas/sociais/populacao/22827-censo-demografico-2022.html"
URL_SISVAN_API = "https://apidadosabertos.saude.gov.br/sisvan/estado-nutricional"
URL_SISVAN_DADOS = "https://dadosabertos.saude.gov.br/dataset/sisvan-estado-nutricional"
URL_MS_NUTRICAO = "https://linhasdecuidado.saude.gov.br/portal/pessoa-idosa/unidade-de-atencao-primaria/avaliacao-antropometrica/"

# Novas APIs adicionadas
URL_DATASUS_SIH = "https://apidadosabertos.saude.gov.br/sih/v1/" # Placeholder - Necessário ajustar para um endpoint específico
URL_NCBI_PUBMED = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
URL_CROSSREF_API = "https://api.crossref.org/"

FONTES = {
    "IBGE SIDRA": URL_IBGE_SIDRA,
    "IBGE Censo 2022": URL_IBGE_CENSO,
    "SISVAN API": URL_SISVAN_API,
    "SISVAN Dados Abertos": URL_SISVAN_DADOS,
    "Ministério da Saúde - Nutrição da pessoa idosa": URL_MS_NUTRICAO,
    "DATASUS SIH": URL_DATASUS_SIH,
    "NCBI PubMed": URL_NCBI_PUBMED,
    "Crossref API": URL_CROSSREF_API
}

def fazer_requisicao(url, params=None):
    resposta = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=TIMEOUT
    )
    resposta.raise_for_status()
    time.sleep(PAUSA_ENTRE_REQUISICOES)
    return resposta

"""# Parte 1 — Coleta por APIs públicas

## 5. IBGE/SIDRA — dados demográficos


In [ ]:
TABELA_IBGE = "9514"
ANO_CENSO = "2022"

# Novo endpoint de exemplo para 'População residente estimada' (Tabela 291)
# n3/all - para todas as grandes regiões
# v/all - para todas as variáveis
# p/2021 - para o ano de 2021 (pode ser ajustado)
URL_POPULACAO = (
    f"{URL_IBGE_SIDRA}/t/{TABELA_IBGE}/"
    f"n1/1/p/{ANO_CENSO}/v/allxp"
)

try:
    resposta_ibge = fazer_requisicao(URL_POPULACAO)
    dados_ibge = resposta_ibge.json()

    if isinstance(dados_ibge, list):
        print("Status:", resposta_ibge.status_code)
        print("Registros recebidos:", len(dados_ibge))
    else:
        print("A resposta do IBGE não está no formato de lista esperado.")
        print("Tipo recebido:", type(dados_ibge).__name__)
        dados_ibge = []

except Exception as erro:
    dados_ibge = []
    print("Não foi possível consultar o SIDRA nesta execução:", erro)

"""## 6. Inspeção e transformação dos dados do IBGE"""

if dados_ibge:
    print("Primeiro registro:")
    print(json.dumps(dados_ibge[0], ensure_ascii=False, indent=2))
else:
    print("A API do IBGE não retornou dados nesta execução.")

"""df_ibge = pd.DataFrame(dados_ibge)

documentos_ibge = []

if not df_ibge.empty:
    # Mantemos os registros da API e acrescentamos uma representação textual.
    # A filtragem temática procura valores de idade que indiquem 60 anos ou mais.
    # Para a tabela 291, as colunas são diferentes. Vamos gerar um texto genérico por registro.

    # Identifica a coluna que contém o valor da população para esta tabela (pode variar)
    # e colunas que descrevem o registro (e.g., 'Local', 'Variável')
    coluna_valor = next(
        (col for col in df_ibge.columns if 'valor' in str(col).lower()),
        None
    )
    colunas_descritivas = [
        col for col in df_ibge.columns
        if col not in ['id', 'D1C', 'D2C', 'D3C', 'D4C', 'V', 'VM', 'D2N', 'D3N', 'D4N', 'NC', 'NN', 'MC', 'MN', 'mu']
    ]

    for indice, registro in df_ibge.iterrows():
        texto_registro_partes = []
        for col in colunas_descritivas:
            if pd.notna(registro[col]) and str(registro[col]).strip() != '':
                texto_registro_partes.append(f"{col}: {registro[col]}")

        if coluna_valor and pd.notna(registro[coluna_valor]):
            texto_registro_partes.append(f"População: {registro[coluna_valor]}")

        texto_registro = " | ".join(texto_registro_partes)

        documentos_ibge.append({
            "id_origem": f"IBGE_291_{indice + 1}",
            "tipo_origem": "SIDRA",
            "categoria": "Demográfico",
            "texto": texto_registro
        })

    print("Registros transformados em documentos:", len(documentos_ibge))
    print("Coluna de valor identificada:", coluna_valor)
else:
    print("Não há dados do IBGE para transformar.")


df_ibge = pd.DataFrame(dados_ibge)

documentos_ibge = []

if not df_ibge.empty:
    # Mantemos os registros da API e acrescentamos uma representação textual.
    # A filtragem temática procura valores de idade que indiquem 60 anos ou mais.
    coluna_idade = next(
        (coluna for coluna in df_ibge.columns
         if "idade" in str(coluna).lower() or "D3N" in str(coluna)),
        None
    )

    for indice, registro in df_ibge.iterrows():
        texto_registro = " | ".join(
            f"{coluna}: {registro[coluna]}"
            for coluna in df_ibge.columns
            if pd.notna(registro[coluna])
        )

        documentos_ibge.append({
            "id_origem": f"IBGE_9514_{indice + 1}",
            "tipo_origem": "SIDRA",
            "categoria": "Demográfico",
            "texto": texto_registro
        })

    print("Registros transformados em documentos:", len(documentos_ibge))
    print("Coluna de idade identificada:", coluna_idade)
else:
    print("Não há dados do IBGE para transformar.")

"""## 7. SISVAN — estado nutricional"""

try:
    resposta_sisvan = fazer_requisicao(URL_SISVAN_API)
    dados_sisvan_api = resposta_sisvan.json()

    print("Status:", resposta_sisvan.status_code)

    if isinstance(dados_sisvan_api, dict):
        print("Chaves recebidas:", list(dados_sisvan_api.keys())[:10])
    elif isinstance(dados_sisvan_api, list):
        print("Registros recebidos:", len(dados_sisvan_api))
    else:
        print("Formato recebido:", type(dados_sisvan_api).__name__)

except Exception as erro:
    dados_sisvan_api = []
    print("A consulta do SISVAN não retornou dados nesta execução:", erro)

# Initialize variables for new APIs to avoid NameError if API calls are skipped or fail
dados_datasus_sih = []
dados_pubmed_api = {}
dados_crossref_api = {}

# Consultando dados de morbidade hospitalar no DATASUS (SIH/SUS)
# ATENÇÃO: O endpoint fornecido é uma base. É necessário adicionar os parâmetros
# específicos para a consulta desejada (ex: /internacoes, filtros por CID-10, etc.).
# O trecho de código para a chamada real da API está comentado abaixo para evitar erros
# e permitir que você especifique os parâmetros corretos para sua consulta.

# try:
#     # Exemplo: Tentar uma consulta genérica ou específica se o endpoint for conhecido
#     # resposta_datasus = fazer_requisicao(URL_DATASUS_SIH + "/algum_endpoint_especifico")
#     # dados_datasus_sih = resposta_datasus.json()
#     # print("Status DATASUS:", resposta_datasus.status_code)
#     # print("Dados DATASUS recebidos (primeiros 500 chars):", str(dados_datasus_sih)[:500])

# except Exception as erro:
#     print("A consulta ao DATASUS não retornou dados nesta execução ou o endpoint requer ajustes:", erro)

# Consultando artigos científicos no NCBI PubMed (Entrez Direct API)
# O trecho de código para a chamada real da API está comentado abaixo.

# try:
#     # 1. Pesquisar IDs de artigos usando MeSH terms
#     termos_busca = "(\"Sarcopenia\"[MeSH Terms] OR \"Sarcopenic Obesity\"[MeSH Terms] OR \"Malnutrition\"[MeSH Terms]) AND \"Aged\"[MeSH Terms]"
#     params_esearch = {
#         "db": "pubmed",
#         "term": termos_busca,
#         "retmax": 5, # Limitar a 5 resultados para este exemplo
#         "retmode": "json"
#     }
#     resposta_esearch = fazer_requisicao(URL_NCBI_PUBMED + "esearch.fcgi", params=params_esearch)
#     ids_pubmed = resposta_esearch.json().get("esearchresult", {}).get("idlist", [])

#     if ids_pubmed:
#         # 2. Obter sumários dos artigos encontrados
#         params_esummary = {
#             "db": "pubmed",
#             "id": ",".join(ids_pubmed),
#             "retmode": "json"
#         }
#         resposta_esummary = fazer_requisicao(URL_NCBI_PUBMED + "esummary.fcgi", params=params_esummary)
#         dados_pubmed_api = resposta_esummary.json().get("result", {})
#         print("Status PubMed:", resposta_esummary.status_code)
#         print("IDs PubMed encontrados:", len(ids_pubmed))
#         if dados_pubmed_api.get('uids'):
#             print("Primeiro artigo PubMed (UID):", dados_pubmed_api['uids'][0])
#     else:
#         print("Nenhum ID encontrado no PubMed para os termos de busca.")

# except Exception as erro:
#     print("A consulta ao NCBI PubMed não retornou dados nesta execução:", erro)

# Consultando metadados de publicações acadêmicas no Crossref
# O trecho de código para a chamada real da API está comentado abaixo.

# try:
#     # Pesquisar artigos relacionados a "sarcopenia obesity"
#     params_crossref = {
#         "query": "sarcopenia obesity",
#         "rows": 5, # Limitar a 5 resultados para este exemplo
#         "sort": "published",
#         "order": "desc"
#     }
#     resposta_crossref = fazer_requisicao(URL_CROSSREF_API + "works", params=params_crossref)
#     dados_crossref_api = resposta_crossref.json()

#     print("Status Crossref:", resposta_crossref.status_code)
#     if dados_crossref_api.get('message', {}).get('total-results') is not None:
#         print("Total de resultados Crossref:", dados_crossref_api['message']['total-results'])
#         if dados_crossref_api['message']['items']:
#             print("Primeiro título Crossref:", dados_crossref_api['message']['items'][0].get('title', ['N/A'])[0])
#     else:
#         print("A API Crossref não retornou 'total-results' ou 'items'.")

# except Exception as erro:
#     print("A consulta ao Crossref API não retornou dados nesta execução:", erro)

# Consultando dados de morbidade hospitalar no DATASUS (SIH/SUS)
# ATENÇÃO: O endpoint fornecido é uma base. É necessário adicionar os parâmetros
# específicos para a consulta desejada (ex: /internacoes, filtros por CID-10, etc.).
# Para evitar erros, inicializamos a variável como lista vazia.
# Exemplo de consulta (pode precisar de ajustes conforme a documentação da API):
# URL_DATASUS_SIH + "internacoes?municipio=...&cid=..."

dados_datasus_sih = []

# try:
#     # Exemplo: Tentar uma consulta genérica ou específica se o endpoint for conhecido
#     # resposta_datasus = fazer_requisicao(URL_DATASUS_SIH + "/algum_endpoint_especifico")
#     # dados_datasus_sih = resposta_datasus.json()
#     # print("Status DATASUS:", resposta_datasus.status_code)
#     # print("Dados DATASUS recebidos (primeiros 500 chars):", str(dados_datasus_sih)[:500])

# except Exception as erro:
#     print("A consulta ao DATASUS não retornou dados nesta execução ou o endpoint requer ajustes:", erro)

# Consultando artigos científicos no NCBI PubMed (Entrez Direct API)

dados_pubmed_api = {}

try:
    # 1. Pesquisar IDs de artigos usando MeSH terms
    termos_busca = "(\"Sarcopenia\"[MeSH Terms] OR \"Sarcopenic Obesity\"[MeSH Terms] OR \"Malnutrition\"[MeSH Terms]) AND \"Aged\"[MeSH Terms]"
    params_esearch = {
        "db": "pubmed",
        "term": termos_busca,
        "retmax": 5, # Limitar a 5 resultados para este exemplo
        "retmode": "json"
    }
    resposta_esearch = fazer_requisicao(URL_NCBI_PUBMED + "esearch.fcgi", params=params_esearch)
    ids_pubmed = resposta_esearch.json().get("esearchresult", {}).get("idlist", [])

    if ids_pubmed:
        # 2. Obter sumários dos artigos encontrados
        params_esummary = {
            "db": "pubmed",
            "id": ",".join(ids_pubmed),
            "retmode": "json"
        }
        resposta_esummary = fazer_requisicao(URL_NCBI_PUBMED + "esummary.fcgi", params=params_esummary)
        dados_pubmed_api = resposta_esummary.json().get("result", {})
        print("Status PubMed:", resposta_esummary.status_code)
        print("IDs PubMed encontrados:", len(ids_pubmed))
        if dados_pubmed_api.get('uids'):
            print("Primeiro artigo PubMed (UID):", dados_pubmed_api['uids'][0])
    else:
        print("Nenhum ID encontrado no PubMed para os termos de busca.")

except Exception as erro:
    print("A consulta ao NCBI PubMed não retornou dados nesta execução:", erro)

# Consultando metadados de publicações acadêmicas no Crossref

dados_crossref_api = {}

try:
    # Pesquisar artigos relacionados a "sarcopenia obesity"
    params_crossref = {
        "query": "sarcopenia obesity",
        "rows": 5, # Limitar a 5 resultados para este exemplo
        "sort": "published",
        "order": "desc"
    }
    resposta_crossref = fazer_requisicao(URL_CROSSREF_API + "works", params=params_crossref)
    dados_crossref_api = resposta_crossref.json()

    print("Status Crossref:", resposta_crossref.status_code)
    if dados_crossref_api.get('message', {}).get('total-results') is not None:
        print("Total de resultados Crossref:", dados_crossref_api['message']['total-results'])
        if dados_crossref_api['message']['items']:
            print("Primeiro título Crossref:", dados_crossref_api['message']['items'][0].get('title', ['N/A'])[0])
    else:
        print("A API Crossref não retornou 'total-results' ou 'items'.")

except Exception as erro:
    print("A consulta ao Crossref API não retornou dados nesta execução:", erro)

"""## 8. Transformação das respostas das APIs em documentos"""

documentos_api = []

# Documento representativo dos dados demográficos do IBGE.
documentos_api.append({
    "id_origem": "IBGE_9514",
    "tipo_origem": "SIDRA",
    "categoria": "Demográfico",
    "texto": (
        "Censo Demográfico 2022. Tabela 9514. População residente por sexo e idade. "
        "Os dados permitem analisar a população brasileira segundo faixas etárias, "
        "com destaque para a população de 60 anos ou mais."
    ),
    "fonte": "IBGE SIDRA",
    "url_origem": URL_POPULACAO
})

# Registros retornados pelo SIDRA.
for documento in documentos_ibge[:100]: # Changed from documentos_ibge to dados_ibge
    documento = documento.copy()
    documento["fonte"] = "IBGE SIDRA"
    documento["url_origem"] = URL_POPULACAO
    documentos_api.append(documento)

# Documento representativo da fonte nutricional do SISVAN.
documentos_api.append({
    "id_origem": "SISVAN_ESTADO_NUTRICIONAL",
    "tipo_origem": "API",
    "categoria": "Nutricional",
    "texto": (
        "SISVAN - Estado Nutricional. Base do Sistema de Vigilância Alimentar "
        "e Nutricional utilizada para acompanhamento do estado nutricional."
    ),
    "fonte": "Ministério da Saúde - SISVAN",
    "url_origem": URL_SISVAN_API
})

# Documento representativo da fonte DATASUS SIH
if dados_datasus_sih:
    documentos_api.append({
        "id_origem": "DATASUS_SIH",
        "tipo_origem": "API",
        "categoria": "Saúde Pública",
        "texto": (
            "DATASUS - Sistema de Informações Hospitalares (SIH/SUS). "
            "Dados de internações por causas relacionadas como desnutrição "
            "e fraturas em idosos."
        ),
        "fonte": "DATASUS SIH",
        "url_origem": URL_DATASUS_SIH
    })

# Documento representativo da fonte NCBI PubMed
if dados_pubmed_api:
    pm_text = f"NCBI PubMed (eUtils). Artigos sobre Sarcopenia e Envelhecimento. Encontrados {len(dados_pubmed_api.get('esearchresult', {}).get('idlist', []))} IDs de artigos."
    if dados_pubmed_api.get("result", {}).get("uids"):
        # Adiciona títulos dos primeiros artigos para melhor descrição
        summaries = [v for k, v in dados_pubmed_api['result'].items() if k != 'uids']
        pm_text += "\n" + " ".join([s.get("title", "") for s in summaries[:5] if s.get("title")]) # add first 5 titles if available

    documentos_api.append({
        "id_origem": "NCBI_PUBMED",
        "tipo_origem": "API",
        "categoria": "Científico",
        "texto": pm_text,
        "fonte": "NCBI PubMed",
        "url_origem": URL_NCBI_PUBMED
    })

# Documento representativo da fonte Crossref API
if dados_crossref_api and dados_crossref_api.get('message', {}).get('items'):
    cr_text = f"Crossref API. Metadados de publicações acadêmicas sobre Sarcopenia e Envelhecimento. Total de {dados_crossref_api['message']['total-results']} resultados."
    for item in dados_crossref_api['message']['items'][:5]: # add first 5 titles if available
        if item.get('title'):
            cr_text += f"\n - {item['title'][0]}"

    documentos_api.append({
        "id_origem": "CROSSREF_API",
        "tipo_origem": "API",
        "categoria": "Científico",
        "texto": cr_text,
        "fonte": "Crossref API",
        "url_origem": URL_CROSSREF_API
    })

print("Documentos textuais derivados das APIs:", len(documentos_api))

"""## 9. Salvamento dos dados brutos"""

pasta_ibge = raiz / "dados_brutos" / "ibge"
pasta_sisvan = raiz / "dados_brutos" / "sisvan"
pasta_datasus = raiz / "dados_brutos" / "datasus"
pasta_pubmed = raiz / "dados_brutos" / "pubmed"
pasta_crossref = raiz / "dados_brutos" / "crossref"

with open(pasta_ibge / "resposta_sidra_9514_2022.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados_ibge, arquivo, ensure_ascii=False, indent=2)

with open(pasta_sisvan / "resposta_sisvan_api.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados_sisvan_api, arquivo, ensure_ascii=False, indent=2)

# Salvar dados DATASUS
if dados_datasus_sih:
    with open(pasta_datasus / "resposta_datasus_sih.json", "w", encoding="utf-8") as arquivo:
        json.dump(dados_datasus_sih, arquivo, ensure_ascii=False, indent=2)

# Salvar dados NCBI PubMed
if dados_pubmed_api:
    with open(pasta_pubmed / "resposta_pubmed_eutils.json", "w", encoding="utf-8") as arquivo:
        json.dump(dados_pubmed_api, arquivo, ensure_ascii=False, indent=2)

# Salvar dados Crossref
if dados_crossref_api:
    with open(pasta_crossref / "resposta_crossref_api.json", "w", encoding="utf-8") as arquivo:
        json.dump(dados_crossref_api, arquivo, ensure_ascii=False, indent=2)

for indice, documento in enumerate(documentos_api, start=1):
    nome_arquivo = f"api_envelhecimento_{indice:04d}.txt"
    if documento["categoria"] == "Nutricional" and "SISVAN" in documento["fonte"]:
        pasta_destino = pasta_sisvan
    elif documento["categoria"] == "Demográfico":
        pasta_destino = pasta_ibge
    elif documento["categoria"] == "Saúde Pública":
        pasta_destino = pasta_datasus
    elif documento["categoria"] == "Científico" and "PubMed" in documento["fonte"]:
        pasta_destino = pasta_pubmed
    elif documento["categoria"] == "Científico" and "Crossref" in documento["fonte"]:
        pasta_destino = pasta_crossref
    else:
        pasta_destino = raiz / "dados_brutos" / "outros" # Fallback para novas categorias
        pasta_destino.mkdir(parents=True, exist_ok=True)

    (pasta_destino / nome_arquivo).write_text(documento["texto"], encoding="utf-8")
    documento["arquivo"] = nome_arquivo

print("Dados brutos das APIs salvos.")

"""# Parte 2 — Coleta por web scraping

## 10. Páginas oficiais


In [ ]:
paginas_oficiais = {
    "SISVAN Dados Abertos": URL_SISVAN_DADOS
}

print("Quantidade de páginas:", len(paginas_oficiais))
for nome, url in paginas_oficiais.items():
    print("-", nome, "->", url)

"""## 11. Verificação de robots.txt"""

def permitido_por_robots(url, user_agent=HEADERS["User-Agent"]):
    partes = urlparse(url)
    robots_url = f"{partes.scheme}://{partes.netloc}/robots.txt"

    analisador = RobotFileParser()
    analisador.set_url(robots_url)

    try:
        analisador.read()
        return analisador.can_fetch(user_agent, url)
    except Exception:
        # Se robots.txt não puder ser consultado, não interrompemos o notebook.
        return True

"""## 12. Função de web scraping"""

def extrair_texto_pagina(url, maximo_paragrafos=30):
    if not permitido_por_robots(url):
        raise PermissionError(f"Coleta não permitida pelo robots.txt: {url}")

    resposta = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")

    titulo = soup.find("h1")
    titulo = titulo.get_text(" ", strip=True) if titulo else ""

    paragrafos = []

    for elemento in soup.find_all(["p", "li"]):
        texto = elemento.get_text(" ", strip=True)
        texto = re.sub(r"\[[^\]]*\]", " ", texto)
        texto = re.sub(r"\s+", " ", texto).strip()

        if len(texto) >= 50:
            paragrafos.append(texto)

        if len(paragrafos) >= maximo_paragrafos:
            break

    return {
        "titulo": titulo,
        "texto": "\n".join(paragrafos),
        "url": resposta.url,
        "status_http": resposta.status_code
    }

"""## 13. Execução do web scraping"""

documentos_scraping = []
erros_scraping = []

for nome_fonte, url in paginas_oficiais.items():
    try:
        resultado = extrair_texto_pagina(url)

        if not resultado["texto"].strip():
            raise ValueError("A página não retornou texto suficiente.")

        if "Ministério da Saúde" in nome_fonte:
            categoria = "Normativa"
        elif "SISVAN" in nome_fonte:
            categoria = "Nutricional"
        else:
            categoria = "Demográfico"

        documento = {
            "fonte": nome_fonte,
            "categoria": categoria,
            "titulo": resultado["titulo"],
            "texto": resultado["texto"],
            "url_origem": resultado["url"],
            "status_http": resultado["status_http"]
        }

        documentos_scraping.append(documento)
        print("Coletado:", nome_fonte)

    except Exception as erro:
        erros_scraping.append({
            "fonte": nome_fonte,
            "url": url,
            "erro": str(erro)
        })
        print("Erro:", nome_fonte, "-", erro)

    time.sleep(PAUSA_ENTRE_REQUISICOES)

print("\nColetas concluídas:", len(documentos_scraping))
print("Erros:", len(erros_scraping))

"""## 14. Inspeção dos documentos coletados"""

if documentos_scraping:
    documento = documentos_scraping[0]
    print("Fonte:", documento["fonte"])
    print("Categoria:", documento["categoria"])
    print("Título:", documento["titulo"])
    print("URL:", documento["url_origem"])
    print("\nTexto:")
    print(documento["texto"][:1500])
else:
    print("Nenhum documento de web scraping foi coletado.")

"""## 15. Salvamento do scraping"""

pasta_scraping = raiz / "dados_brutos" / "web_scraping"

for indice, documento in enumerate(documentos_scraping, start=1):
    nome_arquivo = f"web_envelhecimento_{indice:04d}.txt"
    (pasta_scraping / nome_arquivo).write_text(documento["texto"], encoding="utf-8")
    documento["arquivo"] = nome_arquivo

with open(pasta_scraping / "erros_scraping.json", "w", encoding="utf-8") as arquivo:
    json.dump(erros_scraping, arquivo, ensure_ascii=False, indent=2)

print("Arquivos de web scraping:", len(documentos_scraping))

"""# Parte 3 — Organização e processamento do corpus

## 16. Localização dos documentos


arquivos_brutos = sorted((raiz / "dados_brutos").rglob("*.txt"))

print("Total de documentos brutos:", len(arquivos_brutos))

for arquivo in arquivos_brutos[:10]:
    print("-", arquivo)

"""## 17. Leitura e tratamento de codificação"""

def ler_texto(caminho):
    for codificacao in ["utf-8", "latin-1"]:
        try:
            return caminho.read_text(encoding=codificacao), codificacao, None
        except UnicodeDecodeError:
            continue
        except Exception as erro:
            return None, None, str(erro)

    return None, None, "Codificação não reconhecida."

documentos_lidos = []

for arquivo in arquivos_brutos:
    texto, codificacao, erro = ler_texto(arquivo)

    documentos_lidos.append({
        "arquivo_origem": arquivo,
        "nome_arquivo": arquivo.name,
        "tipo_coleta": arquivo.parent.name,
        "texto_original": texto,
        "codificacao": codificacao,
        "erro": erro
    })

print("Documentos lidos:", len(documentos_lidos))

"""## 18. Normalização textual

## 19. Validação dos documentos


In [ ]:
TAMANHO_MINIMO = 80

for documento in documentos_lidos:
    # Adiciona a chave 'texto_processado' se ela não existir, usando 'texto_original' como fallback
    if "texto_processado" not in documento:
        documento["texto_processado"] = documento["texto_original"]

    texto = documento["texto_processado"]

    if documento["erro"] is not None:
        status = "erro_leitura"
    elif not texto:
        status = "vazio"
    elif len(texto) < TAMANHO_MINIMO:
        status = "muito_curto"
    else:
        status = "valido"

    documento["status"] = status

print(pd.Series([d["status"] for d in documentos_lidos]).value_counts())

"""## 20. Hash dos documentos"""

def calcular_hash(texto):
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

for documento in documentos_lidos:
    texto = documento["texto_processado"] or ""
    documento["hash_sha256"] = calcular_hash(texto)

print("Hashes calculados.")

"""## 21. Deduplicação"""

hashes_encontrados = {}

for documento in documentos_lidos:
    documento["duplicado_de"] = None

    if documento["status"] != "valido":
        continue

    hash_documento = documento["hash_sha256"]

    if hash_documento in hashes_encontrados:
        documento["duplicado_de"] = hashes_encontrados[hash_documento]
        documento["status"] = "duplicado"
    else:
        hashes_encontrados[hash_documento] = documento["nome_arquivo"]

print(
    "Documentos duplicados:",
    sum(d["status"] == "duplicado" for d in documentos_lidos)
)

"""## 22. Salvamento dos documentos processados"""

pasta_processados = raiz / "dados_processados"

for arquivo in pasta_processados.glob("*.txt"):
    arquivo.unlink()

contador = 1

for documento in documentos_lidos:
    if documento["status"] != "valido":
        continue

    nome_processado = f"doc_{contador:04d}.txt"
    caminho = pasta_processados / nome_processado

    caminho.write_text(
        documento["texto_processado"],
        encoding="utf-8"
    )

    documento["arquivo_processado"] = nome_processado
    contador += 1

print("Documentos processados:", contador - 1)

"""## 23. Geração dos metadados"""

registros = []

for indice, documento in enumerate(documentos_lidos, start=1):
    texto = documento["texto_processado"] or ""

    registros.append({
        "id": f"IDOSO_{indice:04d}",
        "arquivo_original": documento["nome_arquivo"],
        "arquivo_processado": documento.get("arquivo_processado"),
        "tipo_coleta": documento["tipo_coleta"],
        "codificacao": documento["codificacao"],
        "status": documento["status"],
        "duplicado_de": documento["duplicado_de"],
        "numero_caracteres": len(texto),
        "numero_palavras": len(texto.split()),
        "hash_sha256": documento["hash_sha256"]
    })

metadados = pd.DataFrame(registros)
metadados.head()

"""## 24. Metadados específicos das fontes"""

metadados_fontes = {}

for documento in documentos_api:
    if "arquivo" in documento:
        metadados_fontes[documento["arquivo"]] = {
            "fonte": documento["fonte"],
            "url_origem": documento["url_origem"],
            "categoria": documento["categoria"],
            "titulo": documento.get("titulo", "")
        }

for documento in documentos_scraping:
    if "arquivo" in documento:
        metadados_fontes[documento["arquivo"]] = {
            "fonte": documento["fonte"],
            "url_origem": documento["url_origem"],
            "categoria": documento["categoria"],
            "titulo": documento.get("titulo", "")
        }

metadados["fonte"] = metadados["arquivo_original"].map(
    lambda n: metadados_fontes.get(n, {}).get("fonte")
)
metadados["url_origem"] = metadados["arquivo_original"].map(
    lambda n: metadados_fontes.get(n, {}).get("url_origem")
)
metadados["categoria"] = metadados["arquivo_original"].map(
    lambda n: metadados_fontes.get(n, {}).get("categoria")
)

metadados

"""## 25. Exportação dos metadados"""

pasta_metadados = raiz / "metadados"

caminho_csv = pasta_metadados / "metadados_corpus_envelhecimento.csv"
caminho_json = pasta_metadados / "metadados_corpus_envelhecimento.json"

metadados.to_csv(caminho_csv, index=False, encoding="utf-8")
metadados.to_json(
    caminho_json,
    orient="records",
    force_ascii=False,
    indent=2
)

print(caminho_csv)
print(caminho_json)

"""## 26. Estatísticas do corpus"""

print("Total de documentos brutos:", len(metadados))
print("Documentos válidos:", (metadados["status"] == "valido").sum())
print("Documentos duplicados:", (metadados["status"] == "duplicado").sum())
print("Documentos vazios:", (metadados["status"] == "vazio").sum())
print("Documentos muito curtos:", (metadados["status"] == "muito_curto").sum())

print("\nDocumentos por categoria:")
print(metadados["categoria"].value_counts(dropna=False))

plt.figure(figsize=(9, 5))
metadados["categoria"].fillna("Sem categoria").value_counts().plot(kind="bar")
plt.title("Distribuição dos documentos por categoria temática")
plt.xlabel("Categoria")
plt.ylabel("Quantidade de documentos")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

"""## 27. Validação do corpus processado"""

erros_validacao = []

for arquivo in sorted(pasta_processados.glob("*.txt")):
    try:
        texto = arquivo.read_text(encoding="utf-8")

        if not texto.strip():
            erros_validacao.append({
                "arquivo": arquivo.name,
                "erro": "Arquivo vazio"
            })

    except Exception as erro:
        erros_validacao.append({
            "arquivo": arquivo.name,
            "erro": str(erro)
        })

print("Arquivos processados:", len(list(pasta_processados.glob("*.txt"))))
print("Erros de validação:", len(erros_validacao))

if erros_validacao:
    print(pd.DataFrame(erros_validacao))
else:
    print("Todos os arquivos processados foram validados com sucesso.")

"""## 28. Consolidação do corpus"""

corpus_completo = []

for arquivo in sorted(pasta_processados.glob("*.txt")):
    corpus_completo.append({
        "arquivo": arquivo.name,
        "texto": arquivo.read_text(encoding="utf-8")
    })

dataframe_corpus = pd.DataFrame(corpus_completo)

caminho_corpus_csv = raiz / "resultados" / "corpus_envelhecimento.csv"
caminho_corpus_json = raiz / "resultados" / "corpus_envelhecimento.json"

dataframe_corpus.to_csv(
    caminho_corpus_csv,
    index=False,
    encoding="utf-8"
)

dataframe_corpus.to_json(
    caminho_corpus_json,
    orient="records",
    force_ascii=False,
    indent=2
)

print("Documentos no corpus final:", len(dataframe_corpus))
print(caminho_corpus_csv)
print(caminho_corpus_json)

"""## 29. Pré-processamento e análise de PLN"""

# Definição global da lista de stopwords em português e da função tokenizar
stopwords_pt = set(stopwords.words('portuguese'))

def tokenizar(texto):
    return word_tokenize(texto.lower(), language='portuguese')

"""### 29.1 Exemplo de tokenização"""

df_nlp = dataframe_corpus.copy()

# Merge com metadados para incluir a coluna 'categoria'
# Assumimos que 'dataframe_corpus' tem uma coluna 'arquivo' que corresponde a 'arquivo_processado' em 'metadados'.
if not metadados.empty and 'categoria' in metadados.columns:
    df_nlp = df_nlp.merge(
        metadados[['arquivo_processado', 'categoria']],
        left_on='arquivo',
        right_on='arquivo_processado',
        how='left'
    ).drop(columns=['arquivo_processado'])
    # Preencher valores NaN na categoria, se houver, para evitar erros futuros
    df_nlp['categoria'] = df_nlp['categoria'].fillna('Não classificado')
else:
    print("Atenção: Metadados não disponíveis ou sem coluna 'categoria'. 'df_nlp' pode não ter categorias.")

if not df_nlp.empty:
    # Pegar o primeiro texto do DataFrame de NLP para demonstração
    exemplo_texto = df_nlp['texto'].iloc[0]
    exemplo_tokens = tokenizar(exemplo_texto)

    print("Texto Original:")
    print(exemplo_texto)
    print("\nTokens (com stopwords e minúsculas):")
    print(exemplo_tokens[:20]) # Mostrar os primeiros 20 tokens para brevidade
else:
    print("O DataFrame 'df_nlp' está vazio, não há textos para tokenizar.")

"""## 30. Exportação dos resultados

## 31. Compactação do projeto


arquivo_zip = shutil.make_archive(
    "EnvelhecimentoPopulacionalBrasil",
    "zip",
    root_dir=raiz
)

print("Arquivo criado:", arquivo_zip)

"""## 32. Referências bibliográficas

1. **IBGE — Instituto Brasileiro de Geografia e Estatística.** Censo Demográfico 2022 e SIDRA.
2. **Ministério da Saúde.** Sistema de Vigilância Alimentar e Nutricional (SISVAN).
3. FREITAS, E. V.; PY, L. *Tratado de Geriatria e Gerontologia*. 5. ed. Guanabara Koogan, 2022.
4. NERI, A. L. Estudos sobre envelhecimento e velhice.

## 33. Especificação metodológica do PLN

Esta seção consolida os critérios metodológicos do segundo notebook sem duplicar o código do pipeline já existente acima.

### 33.1 Caracterização do corpus
O corpus reúne documentos relacionados a **envelhecimento populacional**, **população 60+**, **estado nutricional**, **alimentação e nutrição**, **saúde** e **políticas públicas**. As categorias utilizadas no projeto incluem documentos **Demográficos**, **Nutricionais** e **Normativos**, além de conteúdo institucional quando identificado.

### Pré-processamento
Pipeline: **texto bruto → normalização → tokenização → remoção de ruído → stopwords → lematização**. A tokenização e a lematização utilizam o modelo `pt_core_news_sm` do spaCy, com `token.text` e `token.lemma_`. O stemming RSLP permanece disponível como comparação didática. A representação principal preserva acentos para não perder informação lexical em termos como *nutrição*, *atenção* e *população*.

### Separação das normativas
A análise mantém separado o vocabulário associado à linguagem normativa e o restante do corpus. Essa separação permite observar termos característicos sem misturar automaticamente documentos normativos e não normativos.

### Representação e classificação
A representação automática utiliza **TF-IDF**. A classificação supervisionada permanece disponível apenas como etapa condicional: ela só é executada quando há exemplos suficientes por classe. Como o corpus atual é pequeno, a avaliação principal é feita sobre a **busca textual por TF-IDF**, comparada a um baseline lexical.

### Busca textual
Como complemento à classificação, a busca por similaridade de cosseno sobre TF-IDF recupera documentos mais próximos de uma consulta temática.

### Vieses e limitações
A categorização por origem do documento não equivale a uma anotação humana perfeita. Um corpus pequeno reduz a capacidade de generalização; a disponibilidade de páginas e bases oficiais pode variar; e frequência de palavras não deve ser interpretada automaticamente como importância temática. Os resultados representam o corpus coletado no projeto, não a totalidade da produção textual sobre envelhecimento e nutrição no Brasil.

### Checklist dos requisitos de PLN
- Construção do corpus: contemplada.
- Origem e características dos dados: documentadas.
- Categorias/classes: contempladas.
- Limpeza e normalização: contempladas.
- Tokenização, stopwords e stemming: contemplados.
- TF-IDF: contemplado.
- Busca textual por similaridade: contemplada e avaliada.
- Baseline lexical: contemplado.
- Métricas da busca: Precision@1, Precision@3, Recall@3 e MRR.
- Análise dos erros de recuperação: contemplada.
- Classificação supervisionada: mantida como etapa condicional, pois o corpus atual não sustenta uma avaliação confiável.
- Visualizações: contempladas.

# 33. Avaliação da busca textual: abordagem → resultado → métrica → baseline → erros

Como o corpus final disponível para PLN contém apenas **5 documentos**, a classificação
supervisionada não é estatisticamente adequada para sustentar uma avaliação confiável.
Por isso, a abordagem principal avaliada nesta seção é a **busca textual por TF-IDF e
similaridade de cosseno**, que já é executada no notebook.

Para fechar o ciclo de avaliação, serão comparados:

- **Modelo:** TF-IDF + similaridade de cosseno;
- **Baseline:** sobreposição simples de palavras da consulta com o documento;
- **Relevância:** um documento é considerado relevante quando sua categoria coincide
  com a categoria temática esperada para a consulta.

As métricas utilizadas são **Precision@1, Precision@3, Recall@3 e MRR
(Mean Reciprocal Rank)**. A relevância por categoria é uma anotação fraca, baseada
nos metadados do corpus, e deve ser interpretada como uma avaliação exploratória,
não como um julgamento humano independente.


In [ ]:
# 33.1 Definindo consultas de avaliação e julgamentos de relevância

consultas_avaliacao = [
    {
        "consulta": "população envelhecimento pessoas 60 anos idade",
        "categoria_relevante": "Demográfico"
    },
    {
        "consulta": "estado nutricional alimentação nutrição idosos",
        "categoria_relevante": "Nutricional"
    },
    {
        "consulta": "sarcopenia envelhecimento saúde artigos científicos",
        "categoria_relevante": "Científico"
    }
]

print("Consultas utilizadas na avaliação:")
for item in consultas_avaliacao:
    print(f"- {item['consulta']} -> {item['categoria_relevante']}")

# 33.2 Baseline: busca por sobreposição de palavras

def score_baseline(consulta, texto):
    tokens_consulta = set(tokenizar(consulta))
    tokens_documento = set(tokenizar(texto))
    tokens_consulta = {t for t in tokens_consulta if t.isalpha() and t not in stopwords_pt}
    tokens_documento = {t for t in tokens_documento if t.isalpha() and t not in stopwords_pt}

    if not tokens_consulta:
        return 0.0

    # Proporção de termos da consulta encontrados no documento.
    return len(tokens_consulta.intersection(tokens_documento)) / len(tokens_consulta)


def ranking_tfidf(consulta):
    vetor_consulta = vectorizer.transform([consulta])
    scores = cosine_similarity(vetor_consulta, X).ravel()

    resultado = df_nlp[["arquivo", "categoria", "texto"]].copy()
    resultado["score"] = scores
    resultado["metodo"] = "TF-IDF + cosseno"

    return resultado.sort_values(
        ["score", "arquivo"], ascending=[False, True]
    ).reset_index(drop=True)


def ranking_baseline(consulta):
    resultado = df_nlp[["arquivo", "categoria", "texto"]].copy()
    resultado["score"] = resultado["texto"].apply(
        lambda texto: score_baseline(consulta, texto)
    )
    resultado["metodo"] = "Baseline - sobreposição"

    return resultado.sort_values(
        ["score", "arquivo"], ascending=[False, True]
    ).reset_index(drop=True)


print("Baseline definido: sobreposição de termos da consulta com cada documento.")

# Redefining ranking functions locally to ensure correct behavior after corpus amplification
# and potential redefinitions in other cells.
def score_baseline(consulta, texto):
    tokens_consulta = set(tokenizar(consulta))
    tokens_documento = set(tokenizar(texto))
    tokens_consulta = {t for t in tokens_consulta if t.isalpha() and t not in stopwords_pt}
    tokens_documento = {t for t in tokens_documento if t.isalpha() and t not in stopwords_pt}

    if not tokens_consulta:
        return 0.0

    # Proporção de termos da consulta encontrados no documento.
    return len(tokens_consulta.intersection(tokens_documento)) / len(tokens_consulta)


def ranking_tfidf(consulta):
    # Ensure vectorizer and X are available from the expanded corpus processing
    if "vectorizer" not in globals() or "X" not in globals():
        raise RuntimeError("Vectorizer ou matriz TF-IDF (X) não encontrados. Por favor, execute a etapa de processamento TF-IDF.")

    vetor_consulta = vectorizer.transform([consulta])
    scores = cosine_similarity(vetor_consulta, X).ravel()

    # Ensure df_nlp has the necessary columns. 'arquivo' should have been added by the previous fix.
    resultado = df_nlp[["arquivo", "categoria", "texto"]].copy()
    resultado["score"] = scores
    resultado["metodo"] = "TF-IDF + cosseno"

    return resultado.sort_values(
        ["score", "arquivo"], ascending=[False, True]
    ).reset_index(drop=True)


def ranking_baseline(consulta):
    resultado = df_nlp[["arquivo", "categoria", "texto"]].copy()
    resultado["score"] = resultado["texto"].apply(
        lambda texto: score_baseline(consulta, texto)
    )
    resultado["metodo"] = "Baseline - sobreposição"

    return resultado.sort_values(
        ["score", "arquivo"], ascending=[False, True]
    ).reset_index(drop=True)

def calcular_metricas_busca(ranking, categoria_relevante, k=3):

    # Garante que o ranking tenha índice sequencial
    ranking = ranking.reset_index(drop=True)

    # Verifica quais documentos pertencem à categoria esperada
    relevantes = (
        ranking["categoria"] == categoria_relevante
    ).astype(int).tolist()

    # Precision@1
    precision_1 = relevantes[0] if relevantes else 0.0

    # Precision@K
    top_k = relevantes[:k]

    precision_k = (
        sum(top_k) / len(top_k)
        if top_k else 0.0
    )

    # Recall@K
    total_relevantes = sum(relevantes)

    recall_k = (
        sum(top_k) / total_relevantes
        if total_relevantes > 0
        else 0.0
    )

    # MRR
    reciprocal_rank = 0.0

    for posicao, relevante in enumerate(relevantes, start=1):
        if relevante == 1:
            reciprocal_rank = 1 / posicao
            break

    return {
        "Precision@1": precision_1,
        f"Precision@{k}": precision_k,
        f"Recall@{k}": recall_k,
        "MRR": reciprocal_rank
    }


resultados_avaliacao_busca = []
detalhes_erros_busca = []


# Verificação das variáveis necessárias
if "consultas_avaliacao" not in globals():
    print("ERRO: consultas_avaliacao não foi definida.")

elif "df_nlp" not in globals() or df_nlp.empty:
    print("ERRO: df_nlp não está disponível ou está vazio.")

else:
    # FIX: Ensure 'arquivo' column exists in df_nlp
    if 'arquivo' not in df_nlp.columns:
        # Generate a simple unique identifier for each document using the DataFrame index
        df_nlp['arquivo'] = [f"doc_{i:04d}" for i in df_nlp.index]
        print("INFO: 'arquivo' column added to df_nlp for evaluation.")

    for item in consultas_avaliacao:

        consulta = item["consulta"]
        categoria = item["categoria_relevante"]

        rankings = [
            ("TF-IDF + cosseno", ranking_tfidf(consulta)),
            ("Baseline - sobreposição", ranking_baseline(consulta))
        ]

        for metodo, ranking in rankings:

            # Garante índice correto
            ranking = ranking.reset_index(drop=True)

            # Calcula métricas
            metricas = calcular_metricas_busca(
                ranking,
                categoria,
                k=3
            )

            resultados_avaliacao_busca.append({
                "consulta": consulta,
                "categoria_relevante": categoria,
                "metodo": metodo,
                **metricas
            })

            # Analisa os 3 primeiros resultados
            top_k = ranking.head(3).copy()

            top_k["relevante"] = (
                top_k["categoria"] == categoria
            )

            for posicao, row in top_k.iterrows():

                if not row["relevante"]:

                    detalhes_erros_busca.append({
                        "consulta": consulta,
                        "categoria_relevante": categoria,
                        "metodo": metodo,
                        "posicao": posicao + 1,
                        "arquivo": row["arquivo"],
                        "categoria_recuperada": row["categoria"],
                        "score": row["score"],
                        "tipo_erro": "falso positivo"
                    })


            # Documentos relevantes que ficaram fora do Top-3
            relevantes_fora_top_k = ranking.iloc[3:]

            relevantes_fora_top_k = relevantes_fora_top_k[
                relevantes_fora_top_k["categoria"] == categoria
            ]

            for posicao, row in relevantes_fora_top_k.iterrows():

                detalhes_erros_busca.append({
                    "consulta": consulta,
                    "categoria_relevante": categoria,
                    "metodo": metodo,
                    "posicao": posicao + 1,
                    "arquivo": row["arquivo"],
                    "categoria_recuperada": row["categoria"],
                    "score": row["score"],
                    "tipo_erro": "falso negativo fora do top-3"
                })


    # DataFrames finais
    avaliacao_busca = pd.DataFrame(
        resultados_avaliacao_busca
    )

    erros_busca = pd.DataFrame(
        detalhes_erros_busca
    )


    # Mostrar resultados
    print("Resultados agregados:")

    display(
        avaliacao_busca
        .groupby("metodo")[
            [
                "Precision@1",
                "Precision@3",
                "Recall@3",
                "MRR"
            ]
        ]
        .mean()
        .round(3)
 )

"""### 33.4 Interpretação das métricas

- **Precision@1:** verifica se o primeiro documento recuperado é relevante.
- **Precision@3:** mede a proporção de documentos relevantes entre os três primeiros resultados.
- **Recall@3:** mede quanto dos documentos relevantes disponíveis foram recuperados entre os três primeiros.
- **MRR:** considera a posição do primeiro documento relevante; quanto mais cedo ele aparece,
  maior o valor.

A comparação com o baseline permite verificar se a representação TF-IDF acrescenta valor
em relação a uma estratégia simples de correspondência lexical.


# 33.5 Comparação visual entre modelo e baseline

metricas_medias = (
    avaliacao_busca
    .groupby("metodo")[["Precision@1", "Precision@3", "Recall@3", "MRR"]]
    .mean()
)

ax = metricas_medias.plot(kind="bar", figsize=(10, 5))
ax.set_title("Busca TF-IDF versus baseline lexical")
ax.set_ylabel("Pontuação média")
ax.set_xlabel("Método")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

# 33.6 Visualização dos erros de recuperação

if not erros_busca.empty:
    contagem_erros = (
        erros_busca
        .groupby(["metodo", "tipo_erro"])
        .size()
        .unstack(fill_value=0)
    )

    contagem_erros.plot(kind="bar", figsize=(10, 5))
    plt.title("Erros de recuperação no Top-3")
    plt.ylabel("Quantidade de ocorrências")
    plt.xlabel("Método")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum erro de recuperação foi identificado no Top-3.")

print("\nDetalhamento dos erros:")
if erros_busca.empty:
    print("Nenhum erro encontrado.")
else:
    display(
        erros_busca[
            [
                "consulta", "categoria_relevante", "metodo",
                "posicao", "arquivo", "categoria_recuperada",
                "score", "tipo_erro"
            ]
        ].sort_values(["consulta", "metodo", "posicao"])
    )

"""### 33.7 Análise dos erros

Os erros de recuperação devem ser interpretados de acordo com o tamanho e a composição
do corpus. Um falso positivo ocorre quando um documento de outra categoria aparece no
Top-3. Um falso negativo ocorre quando um documento da categoria relevante fica fora do
Top-3.

No corpus atual, a semelhança lexical pode favorecer documentos que compartilham termos
genéricos, como *envelhecimento*, *saúde*, *população* ou *idosos*. O TF-IDF reduz parcialmente
o peso de termos muito frequentes, mas não resolve a ausência de contexto semântico.

Além disso, a categoria usada como critério de relevância foi atribuída durante a construção
do corpus e não constitui uma anotação humana independente. Portanto, os resultados servem
para avaliar o comportamento do pipeline neste corpus específico, e não para afirmar desempenho
geral de um sistema de recuperação de informação.


In [ ]:
# 33.8 Exportação dos resultados da avaliação

# Define a variável pasta_resultados caso não esteja definida
if "pasta_resultados" not in globals():
    if "raiz" in globals():
        pasta_resultados = raiz / "resultados"
    else:
        from pathlib import Path
        pasta_resultados = Path("EnvelhecimentoPopulacionalBrasil/resultados")

pasta_resultados.mkdir(parents=True, exist_ok=True)

avaliacao_busca.to_csv(
    pasta_resultados / "avaliacao_busca_tfidf_vs_baseline.csv",
    index=False,
    encoding="utf-8"
)

erros_busca.to_csv(
    pasta_resultados / "analise_erros_busca.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivos de avaliação salvos em:")
print(pasta_resultados / "avaliacao_busca_tfidf_vs_baseline.csv")
print(pasta_resultados / "analise_erros_busca.csv")

"""## 34. Checklist dos requisitos

| Requisito | Implementação |
|---|---|
| Seleção/coleta/construção do corpus | API + web scraping |
| Origem e características dos dados | Metadados, fontes e categorias |
| Classes/categorias | Demográfico, Nutricional e Científico no corpus válido |
| Vieses e limitações | Discussão metodológica e seção específica |
| Limpeza e normalização | Unicode, URLs, ruído e espaços |
| Tokenização | `word_tokenize` com fallback |
| Lematização/stemming | Lematização com spaCy; stemming RSLP como comparação |
| Stopwords | Stopwords em português |
| Representação textual | TF-IDF |
| Abordagem de análise | Busca textual por similaridade de cosseno |
| Métricas | Precision@1, Precision@3, Recall@3 e MRR |
| Baseline | Sobreposição lexical de palavras |
| Análise de erros | Falsos positivos e falsos negativos no Top-3 |
| Matriz de confusão | Não aplicável à busca textual; a classificação permanece condicionada ao tamanho do corpus |
| Visualizações | Quatro visualizações principais explicitamente reunidas na seção 36.9, além das visualizações de avaliação |

### Observação metodológica final

A classificação supervisionada não é apresentada como resultado principal porque o corpus final
possui poucos documentos e não apresenta quantidade suficiente de exemplos por classe para uma
avaliação confiável. Em vez de simular uma avaliação supervisionada, este notebook avalia
quantitativamente a **abordagem de busca textual que efetivamente é executada**, comparando-a
com um baseline e documentando seus erros e limitações.

## 35. Síntese metodológica

O foco deste trabalho é a **análise e recuperação de informações sobre envelhecimento populacional e estado nutricional**. A tarefa principal é tratada como busca textual, usando **TF-IDF e similaridade do cosseno**, com comparação a um **baseline lexical**.

A avaliação considera **Precision@1, Precision@3, Recall@3 e MRR**, além da análise de falsos positivos e falsos negativos. A classificação supervisionada permanece apenas como possibilidade complementar, pois o corpus final é pequeno para sustentar uma avaliação robusta dessa tarefa.

**Limitação principal:** ampliar o número de documentos e manter representação equilibrada das categorias fortaleceria a validade da avaliação.

# Parte 4 — Ampliação do corpus por APIs públicas

O corpus inicial possuía apenas 5 documentos, o que é insuficiente para sustentar conclusões amplas sobre desempenho de métodos de PLN. Nesta etapa, o corpus é ampliado por fontes públicas e reprodutíveis relacionadas diretamente ao tema.

Foram incorporadas três frentes de coleta:

1. **IBGE/SIDRA** — dados demográficos, incluindo população por grupo de idade;
2. **SISVAN / Ministério da Saúde** — dados e metadados de estado nutricional;
3. **OpenAlex e Crossref** — metadados de produção científica sobre envelhecimento, nutrição e população idosa.

A coleta é limitada por quantidade, deduplicação e termos de relevância para evitar que o corpus fique artificialmente dominado por uma única fonte.


# Instalação/importação das dependências da etapa de ampliação
!pip -q install requests

import requests
import time
import unicodedata
from collections import defaultdict

def normalizar_chave(texto):
    texto = str(texto or "").lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\\s+", " ", texto).strip()
    return texto

def texto_relevante(texto):
    termos = [
        "envelhecimento", "idoso", "idosos", "população idosa",
        "nutrição", "nutricional", "estado nutricional",
        "alimentação", "sarcopenia", "obesidade", "desnutrição",
        "antropometria", "demografia", "longevidade"
    ]
    t = normalizar_chave(texto)
    return any(term in t for term in [normalizar_chave(x) for x in termos])

"""## 36.1 IBGE/SIDRA — dimensão demográfica

O SIDRA disponibiliza dados por API em JSON. Para este projeto, usamos a tabela **5918 — População, por grupo de idade**, que permite incorporar registros associados à distribuição etária da população. O IBGE documenta o acesso aos dados por API SIDRA. citeturn4search6turn0search12


In [ ]:
# IBGE/SIDRA: população por grupo de idade
# A consulta retorna dados em JSON e cria documentos textuais a partir
# dos rótulos e valores da tabela.

ibge_docs = []

try:
    url_sidra = "https://apisidra.ibge.gov.br/values/t/5918/n1/all/p/all/v/all"
    resp = requests.get(url_sidra, timeout=30)
    resp.raise_for_status()
    dados_sidra = resp.json()

    for row in dados_sidra[1:]:
        grupo = row.get("D3N") or row.get("D3") or ""
        periodo = row.get("D2N") or row.get("D2") or ""
        valor = row.get("V") or ""

        texto = (
            f"IBGE população por grupo de idade. "
            f"Período {periodo}. Grupo de idade {grupo}. "
            f"Valor observado {valor}. "
            f"Indicador demográfico relacionado ao envelhecimento populacional."
        )

        if texto_relevante(texto):
            ibge_docs.append({
                "titulo": f"IBGE/SIDRA - População por grupo de idade - {grupo}",
                "texto": texto,
                "categoria": "Demográfico",
                "fonte": "IBGE/SIDRA API",
                "tipo_fonte": "API"
            })

    print(f"Registros relevantes obtidos do IBGE/SIDRA: {len(ibge_docs)}")

except Exception as e:
    print("Não foi possível consultar o SIDRA nesta execução:", e)
    print("O código permanece no notebook para execução posterior no Colab.")

"""## 36.2 SISVAN — estado nutricional

O Ministério da Saúde disponibiliza uma API de Dados Abertos com o endpoint **`/sisvan/estado-nutricional`**. O portal também disponibiliza arquivos anuais de estado nutricional e informa que a base contém dados anonimizados de acompanhamento antropométrico. citeturn2search0turn2search1

Para o PLN, os registros numéricos são convertidos em pequenos documentos descritivos, preservando a origem e o ano. Isso permite integrar a dimensão nutricional ao corpus sem tratar os dados como se fossem textos científicos.


# Ministério da Saúde / SISVAN
# A API oficial informa o endpoint /sisvan/estado-nutricional.
# Como os parâmetros podem variar, fazemos uma consulta inicial e
# transformamos apenas registros com campos descritivos úteis ao corpus.

sisvan_docs = []

try:
    url_sisvan = "https://apidadosabertos.saude.gov.br/sisvan/estado-nutricional"
    resp = requests.get(url_sisvan, timeout=30)
    resp.raise_for_status()
    dados_sisvan = resp.json()

    registros = dados_sisvan.get("data", dados_sisvan)
    if isinstance(registros, dict):
        registros = [registros]

    for i, row in enumerate(registros[:500]):
        if not isinstance(row, dict):
            continue

        partes = []
        for k, v in row.items():
            if v not in (None, "", "null"):
                partes.append(f"{k}: {v}")

        texto = (
            "SISVAN estado nutricional. "
            + " ".join(partes)
            + " Informação relacionada à vigilância alimentar e nutricional."
        )

        if texto_relevante(texto):
            sisvan_docs.append({
                "titulo": f"SISVAN - Estado nutricional - registro {i+1}",
                "texto": texto,
                "categoria": "Nutricional",
                "fonte": "Ministério da Saúde / SISVAN API",
                "tipo_fonte": "API"
            })

    print(f"Registros relevantes obtidos do SISVAN: {len(sisvan_docs)}")

except Exception as e:
    print("Não foi possível consultar a API SISVAN nesta execução:", e)
    print("O código permanece no notebook para execução posterior no Colab.")

"""## 36.3 OpenAlex — literatura científica

A API do OpenAlex permite consultar trabalhos acadêmicos. Nesta etapa, os resultados são convertidos em documentos contendo título, resumo quando disponível, ano e periódico. A consulta usa termos diretamente relacionados ao tema, aumentando a variedade da categoria científica. citeturn1search1


In [ ]:
# OpenAlex: trabalhos científicos relacionados ao tema
openalex_docs = []

def reconstruir_abstract_inverted(inv):
    if not isinstance(inv, dict):
        return ""
    palavras = []
    for palavra, posicoes in inv.items():
        for pos in posicoes:
            palavras.append((pos, palavra))
    palavras.sort()
    return " ".join(p for _, p in palavras)

consultas_openalex = [
    "population aging nutritional status",
    "older adults nutrition",
    "aging sarcopenia nutrition",
    "elderly nutritional assessment"
]

for consulta in consultas_openalex:
    try:
        url = "https://api.openalex.org/works"
        params = {
            "search": consulta,
            "per-page": 25,
            "select": "id,doi,title,publication_year,primary_location,abstract_inverted_index"
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()

        for item in r.json().get("results", []):
            titulo = item.get("title") or ""
            resumo = reconstruir_abstract_inverted(item.get("abstract_inverted_index"))
            ano = item.get("publication_year") or ""

            texto = f"{titulo}. {resumo}".strip()
            if texto_relevante(texto):
                openalex_docs.append({
                    "titulo": titulo,
                    "texto": texto,
                    "categoria": "Científico",
                    "fonte": "OpenAlex API",
                    "tipo_fonte": "API",
                    "ano": ano,
                    "doi": item.get("doi") or ""
                })

        time.sleep(0.2)

    except Exception as e:
        print(f"Falha na consulta OpenAlex '{consulta}':", e)

print(f"Registros científicos relevantes obtidos do OpenAlex: {len(openalex_docs)}")

"""## 36.4 Crossref — literatura científica complementar

O Crossref oferece uma API REST para consulta de trabalhos por campos bibliográficos. Ela é usada aqui como fonte complementar para ampliar a diversidade da literatura, reduzindo a dependência de uma única base científica. citeturn1search0


# Crossref: literatura científica complementar
crossref_docs = []

consultas_crossref = [
    "population aging nutrition",
    "older adults nutritional status",
    "elderly nutrition Brazil",
    "aging sarcopenia"
]

for consulta in consultas_crossref:
    try:
        url = "https://api.crossref.org/works"
        params = {
            "query.bibliographic": consulta,
            "rows": 20,
            "select": "DOI,title,author,published,container-title,abstract"
        }
        headers = {"User-Agent": "PLN-envelhecimento-projeto/1.0"}

        r = requests.get(url, params=params, headers=headers, timeout=30)
        r.raise_for_status()

        for item in r.json().get("message", {}).get("items", []):
            titulos = item.get("title") or []
            titulo = titulos[0] if titulos else ""
            resumo = re.sub(r"<[^>]+>", " ", item.get("abstract") or "")
            texto = f"{titulo}. {resumo}".strip()

            if texto_relevante(texto):
                crossref_docs.append({
                    "titulo": titulo,
                    "texto": texto,
                    "categoria": "Científico",
                    "fonte": "Crossref API",
                    "tipo_fonte": "API",
                    "doi": item.get("DOI", "")
                })

        time.sleep(0.2)

    except Exception as e:
        print(f"Falha na consulta Crossref '{consulta}':", e)

print(f"Registros científicos relevantes obtidos do Crossref: {len(crossref_docs)}")

"""## 36.5 Consolidação, deduplicação e controle de tamanho

As novas fontes são unificadas com o corpus original. A deduplicação usa título/texto normalizado, evitando que o mesmo trabalho ou registro apareça várias vezes.

A categoria continua baseada no domínio da fonte:
- **Demográfico:** IBGE/SIDRA;
- **Nutricional:** SISVAN/Ministério da Saúde;
- **Científico:** OpenAlex e Crossref;
- **Normativa:** documentos normativos já coletados anteriormente.

O objetivo é aumentar o número de documentos sem perder rastreabilidade da origem.


In [ ]:
# Recupera o corpus original quando df_nlp já existe
originais = []

if "df_nlp" in globals() and isinstance(df_nlp, pd.DataFrame):
    col_texto = "texto" if "texto" in df_nlp.columns else None
    if col_texto:
        for _, row in df_nlp.iterrows():
            originais.append({
                "titulo": row.get("titulo", "Documento original"),
                "texto": row.get(col_texto, ""),
                "categoria": row.get("categoria", "Não classificado"),
                "fonte": row.get("fonte", "Corpus original"),
                "tipo_fonte": row.get("tipo_fonte", "Original")
            })

novos_docs = ibge_docs + sisvan_docs + openalex_docs + crossref_docs
todos_docs = originais + novos_docs

# Deduplicação
vistos = set()
corpus_expandido = []

for doc in todos_docs:
    chave = normalizar_chave(
        doc.get("titulo", "") + " " + doc.get("texto", "")
    )
    if not chave or chave in vistos:
        continue
    vistos.add(chave)
    corpus_expandido.append(doc)

df_nlp = pd.DataFrame(corpus_expandido)

if not df_nlp.empty:
    df_nlp["texto"] = df_nlp["texto"].fillna("").astype(str)
    df_nlp["categoria"] = df_nlp["categoria"].fillna("Não classificado")
    df_nlp["fonte"] = df_nlp["fonte"].fillna("Não informado")

print("Corpus antes da ampliação:", len(originais))
print("Novos registros coletados:", len(novos_docs))
print("Corpus final após deduplicação:", len(df_nlp))

display(
    df_nlp.groupby(["categoria", "fonte"])
          .size()
          .reset_index(name="documentos")
          .sort_values("documentos", ascending=False)
)

if len(df_nlp) < 30:
    print("ATENÇÃO: o corpus ainda está pequeno. Execute novamente no Colab ou aumente os limites de coleta.")
else:
    print("Corpus ampliado para análise de PLN.")

"""## 36.6 Reexecução do TF-IDF

Depois da ampliação, o TF-IDF e a busca por similaridade são reconstruídos sobre o corpus final. Assim, as métricas de recuperação deixam de depender exclusivamente dos 5 documentos iniciais.


# Reprocessamento do corpus ampliado
if not df_nlp.empty:
    df_nlp["texto_limpo"] = df_nlp["texto"].apply(
        lambda x: limpar_texto(x) if "limpar_texto" in globals() else str(x).lower()
    )

    # Mantém as funções de tokenização/stopwords já definidas no notebook.
    if "processar_texto" in globals():
        df_nlp["texto_processado"] = df_nlp["texto"].apply(processar_texto)
    elif "preprocessar_texto" in globals():
        df_nlp["texto_processado"] = df_nlp["texto"].apply(preprocessar_texto)
    else:
        df_nlp["texto_processado"] = df_nlp["texto_limpo"]

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    vectorizer = TfidfVectorizer(
        max_df=0.95,
        min_df=1,
        ngram_range=(1, 2)
    )
    X = vectorizer.fit_transform(df_nlp["texto_processado"].fillna(""))

    print("Documentos:", X.shape[0])
    print("Termos na representação TF-IDF:", X.shape[1])

"""## 36.7 Avaliação da recuperação

As consultas de teste continuam distribuídas entre as três categorias principais. A avaliação usa **Precision@1, Precision@3, Recall@3 e MRR**, comparando TF-IDF com o baseline lexical.

Como os rótulos são derivados das categorias do corpus e não de uma anotação humana independente, os resultados devem ser interpretados como **avaliação experimental do corpus construído**, e não como medida definitiva de qualidade de busca na população real.


In [ ]:
# Avaliação sobre o corpus ampliado
consultas_expandidas = [
    ("população envelhecimento pessoas 60 anos idade", "Demográfico"),
    ("estado nutricional alimentação nutrição idosos", "Nutricional"),
    ("sarcopenia envelhecimento saúde artigos científicos", "Científico"),
    ("idosos avaliação nutricional antropometria", "Nutricional"),
    ("envelhecimento populacional distribuição etária", "Demográfico"),
    ("older adults nutrition aging scientific study", "Científico"),
]

def ranking_tfidf(query):
    q = vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    return np.argsort(scores)[::-1], scores

resultados_ampliados = []

for query, categoria_esperada in consultas_expandidas:
    ordem, scores = ranking_tfidf(query)
    categorias = df_nlp.iloc[ordem]["categoria"].tolist()

    top1 = categorias[:1]
    top3 = categorias[:3]

    p1 = 1.0 if top1 and top1[0] == categoria_esperada else 0.0
    p3 = sum(c == categoria_esperada for c in top3) / 3
    recall3 = min(sum(c == categoria_esperada for c in top3), 1)

    rr = 0.0
    for pos, cat in enumerate(top3, start=1):
        if cat == categoria_esperada:
            rr = 1 / pos
            break

    resultados_ampliados.append({
        "consulta": query,
        "categoria_esperada": categoria_esperada,
        "Precision@1": p1,
        "Precision@3": p3,
        "Recall@3": recall3,
        "MRR": rr,
        "top1": top1[0] if top1 else ""
    })

df_avaliacao_ampliada = pd.DataFrame(resultados_ampliados)

print("Métricas médias no corpus ampliado:")
display(df_avaliacao_ampliada[["Precision@1", "Precision@3", "Recall@3", "MRR"]].mean().to_frame("média"))

display(df_avaliacao_ampliada)

"""## 36.9 Visualizações finais do corpus e dos resultados

Esta seção reúne **seis visualizações distintas e diretamente relacionadas aos requisitos da atividade**.

1. Distribuição dos documentos por categoria;
2. Principais termos da representação TF-IDF;
3. Distribuição do tamanho dos documentos por categoria;
4. Comparação das métricas entre TF-IDF e o baseline lexical;
5. Gráfico de pizza da composição do corpus por categoria;
6. Nuvem de palavras dos termos mais frequentes do corpus.


# 36.9.1 — Distribuição final dos documentos por categoria
if not df_nlp.empty and "categoria" in df_nlp.columns:
    contagem_categorias = df_nlp["categoria"].fillna("Sem categoria").value_counts()
    plt.figure(figsize=(9, 5))
    contagem_categorias.plot(kind="bar")
    plt.title("Distribuição final do corpus por categoria")
    plt.xlabel("Categoria")
    plt.ylabel("Quantidade de documentos")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados suficientes para a visualização de categorias.")

# 36.9.2 — Principais termos da representação TF-IDF
if "X" in globals() and "vectorizer" in globals():
    pesos_medios = np.asarray(X.mean(axis=0)).ravel()
    termos = np.array(vectorizer.get_feature_names_out())
    top_idx = pesos_medios.argsort()[-15:][::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(termos[top_idx][::-1], pesos_medios[top_idx][::-1])
    plt.title("15 principais termos pela média dos pesos TF-IDF")
    plt.xlabel("Peso médio TF-IDF")
    plt.ylabel("Termo")
    plt.tight_layout()
    plt.show()
else:
    print("Representação TF-IDF indisponível para a visualização de termos.")

# 36.9.3 — Tamanho dos documentos por categoria
if not df_nlp.empty:
    df_visual = df_nlp.copy()
    df_visual["tamanho_tokens"] = df_visual["texto"].fillna("").apply(lambda x: len(str(x).split()))
    df_visual["categoria"] = df_visual["categoria"].fillna("Sem categoria")
    df_visual.boxplot(column="tamanho_tokens", by="categoria", figsize=(10, 6), rot=20)
    plt.suptitle("")
    plt.title("Distribuição do tamanho dos documentos por categoria")
    plt.xlabel("Categoria")
    plt.ylabel("Quantidade aproximada de tokens")
    plt.tight_layout()
    plt.show()
else:
    print("Corpus indisponível para a visualização do tamanho dos documentos.")

# 36.9.4 — Comparação das métricas de recuperação entre TF-IDF e baseline
# A avaliação ampliada registra somente o TF-IDF. Para comparar dois métodos,
# usamos a avaliação principal, que contém TF-IDF e o baseline lexical.
if "avaliacao_busca" in globals() and not avaliacao_busca.empty and "metodo" in avaliacao_busca.columns:
    colunas_metricas = [c for c in ["Precision@1", "Precision@3", "Recall@3", "MRR"] if c in avaliacao_busca.columns]
    if colunas_metricas:
        metricas_medias_finais = avaliacao_busca.groupby("metodo")[colunas_metricas].mean()
        ax = metricas_medias_finais.plot(kind="bar", figsize=(10, 5))
        ax.set_title("TF-IDF versus baseline lexical — métricas médias")
        ax.set_xlabel("Método")
        ax.set_ylabel("Pontuação média")
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("As métricas de recuperação não estão disponíveis.")
else:
    print("Avaliação TF-IDF versus baseline ainda não disponível para comparação visual.")

# 36.9.5 — Gráfico de pizza da composição do corpus
if not df_nlp.empty and "categoria" in df_nlp.columns:
    contagem_pizza = df_nlp["categoria"].fillna("Sem categoria").value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(
        contagem_pizza.values,
        labels=contagem_pizza.index,
        autopct="%1.1f%%",
        startangle=90
    )
    plt.title("Composição do corpus por categoria")
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados suficientes para o gráfico de pizza.")

# 36.9.6 — Nuvem de palavras do corpus
if not df_nlp.empty and "texto" in df_nlp.columns:
    try:
        from wordcloud import WordCloud

        texto_corpus = " ".join(df_nlp["texto"].fillna("").astype(str).tolist())
        stopwords_nuvem = set(stopwords_pt) if "stopwords_pt" in globals() else set()

        nuvem = WordCloud(
            width=1200,
            height=600,
            background_color="white",
            stopwords=stopwords_nuvem,
            collocations=False,
            min_font_size=10
        ).generate(texto_corpus)

        plt.figure(figsize=(14, 7))
        plt.imshow(nuvem, interpolation="bilinear")
        plt.axis("off")
        plt.title("Nuvem de palavras do corpus")
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("A biblioteca 'wordcloud' não está disponível neste ambiente.")
else:
    print("Não há textos suficientes para gerar a nuvem de palavras.")

"""## 36.10 Limitações após a ampliação

A ampliação reduz a limitação do corpus de 5 documentos, mas não elimina todas as limitações.

- As fontes possuem naturezas diferentes: dados estatísticos, registros de vigilância nutricional e literatura científica.
- Os registros do SISVAN são dados de acompanhamento nutricional, não textos escritos originalmente para tarefas de PLN.
- OpenAlex e Crossref fornecem metadados científicos; a disponibilidade de resumos varia entre registros.
- As categorias são definidas pela origem/domínio da fonte, portanto não equivalem necessariamente a uma anotação semântica humana.
- As métricas de busca dependem das consultas e dos rótulos adotados.
- O SISVAN informa limitações relacionadas à existência de múltiplos acompanhamentos e às diferentes origens dos registros; essas características devem ser consideradas na interpretação dos dados. citeturn2search1

Portanto, o objetivo do experimento é avaliar a aplicação de técnicas de PLN em um corpus heterogêneo sobre o tema, e não produzir inferências epidemiológicas ou demográficas a partir dos resultados de busca.

# 37. Síntese final do projeto

**Título:** Aplicação de técnicas de Processamento de Linguagem Natural na análise e recuperação de informações sobre envelhecimento populacional e estado nutricional

**Problema de pesquisa:** Como técnicas de representação textual, como TF-IDF, podem ser utilizadas para recuperar documentos relevantes sobre envelhecimento populacional e estado nutricional em um corpus heterogêneo?

**Contribuição metodológica:** o projeto combina fontes públicas, pré-processamento em português, representação TF-IDF, busca por similaridade, baseline lexical, métricas de recuperação e análise de erros.

**Melhoria em relação à versão inicial:** o corpus deixa de depender exclusivamente de 5 documentos e passa a incorporar fontes via APIs públicas, com deduplicação e registro da origem dos documentos.
